# Anatomy slices: CT-like cross-sections from the Z-Anatomy atlas, fitted to a model
Z-Anatomy (CC BY-SA 4.0, BodyParts3D-based) gives bones, muscles, skin and muscle insertions. Each body segment (pelvis,
trunk, neck, head; upper arm, forearm, hand, thigh, calf, foot on both sides; see `model_to_vrm/anatomy/segments.py`) is cut at
its own **critical levels** (peaks and valleys of the skin section's size, off-centre shift and flattening along the
bone, ripples dropped by persistence), then each slice's outline is fitted to the model's own skin
(5 numbers per slice: shift, roll, width, depth); the atlas muscles and bones ride along. At joint zones (hip, shoulder,
crotch) a slice owns only part of the skin ring: slices stop where the ring coverage drops, and the zone between two
segments' end slices is left to a blend.

**Needs:** the model's pipeline run up to step 7 (skeleton, materials, flesh). Kernel `E:\splat\.venv\Scripts\python.exe`.

In [ ]:
import os, sys, glob, subprocess, html, base64
from IPython.display import display, HTML
REPO = r'E:\code\model-to-vrm'
PY = r'E:\splat\.venv\Scripts\python.exe'; PYB = r'E:\splat\.venv-body\Scripts\python.exe'; PYS = r'E:\splat\.venv-sim\Scripts\python.exe'
BLENDER = r'C:\Program Files\Blender Foundation\Blender 5.3\blender.exe'
def sh(cmd, log=None):
    """run a command, print its last lines; stop the notebook on failure"""
    r = subprocess.run(cmd, capture_output=True, text=True, encoding='utf-8', errors='replace', cwd=REPO)
    out = (r.stdout or '') + (r.stderr or '')
    if log: open(log, 'a', encoding='utf-8').write(out)
    print('\n'.join(l for l in out.splitlines()[-15:] if 'it/s]' not in l))
    if r.returncode: raise RuntimeError('failed: %s' % ' '.join(cmd[:4]))
def show(*patterns, width=320):
    """show images side by side (glob patterns), newest render of each file"""
    files = [f for p in patterns for f in sorted(glob.glob(p))]
    if not files: print('no pictures for', patterns); return
    tags = []
    for f in files:
        ext = os.path.splitext(f)[1][1:].replace('jpg', 'jpeg'); b = base64.b64encode(open(f, 'rb').read()).decode()
        tags.append('<figure style="display:inline-block;margin:4px"><img src="data:image/%s;base64,%s" width="%d"><figcaption style="font-size:11px">%s</figcaption></figure>' % (ext, b, width, html.escape(os.path.basename(f))))
    display(HTML(''.join(tags)))

ATLAS_BLEND = r'E:\figures\models\zanatomy\blend\Startup.blend'
ATLAS = r'E:\figures\models\zanatomy\export'      # atlas.npz goes here
SLICES = r'E:\figures\anatomy_test'               # atlas slices per segment
MODEL_DIR = r'E:\figures\model_r'                   # a model already through pipeline.ipynb
sys.path.insert(0, os.path.join(REPO, 'model_to_vrm', 'anatomy')); from segments import ORDER
SEGMENTS = ORDER                                   # every body segment, both sides
A = lambda *p: os.path.join(REPO, 'model_to_vrm', 'anatomy', *p)

## 1. Export the atlas once (bones, muscles, skin, insertions → arrays)

In [ ]:
FORCE = False
if FORCE or not os.path.exists(ATLAS + r'\atlas.npz'):
    sh([BLENDER, '-b', '-Y', ATLAS_BLEND, '-P', A('za_export.py'), '--', ATLAS])
print('atlas ready:', os.path.exists(ATLAS + r'\atlas.npz'))

## 2. Critical slices of each segment (bone tan, muscles red/purple, skin outline)

In [ ]:
FORCE = False
for sg in SEGMENTS:
    if FORCE or not os.path.exists(os.path.join(SLICES, sg + '.npz')):
        sh([PY, '-W', 'ignore', A('za_slices.py'), ATLAS + r'\atlas.npz', sg, SLICES])
    show(os.path.join(SLICES, sg + '_sheet.png'), width=900)

## 3. Fit the slices to the model (dots: the model's surface — pink skin, blue cloth, green prop; red: fitted outline)

In [ ]:
FORCE = False
for sg in SEGMENTS:
    out = os.path.join(MODEL_DIR, 'anatomy')
    if FORCE or not os.path.exists(os.path.join(out, sg + '_fit.npz')):
        sh([PY, '-W', 'ignore', A('fit_slices.py'), MODEL_DIR, sg, os.path.join(SLICES, sg + '.npz'), out])
    show(os.path.join(out, sg + '_fit_sheet.png'), width=900)

## 3b. All fitted slices inside the model, in Blender (red: fitted skin outline, purple: muscles, tan: bones, blue: segment axis)

In [ ]:
FORCE = False
out = os.path.join(MODEL_DIR, 'anatomy'); pre = os.path.join(out, 'slices3d_all')
fits = [os.path.join(out, sg + '_fit.npz') for sg in SEGMENTS if os.path.exists(os.path.join(out, sg + '_fit.npz'))]
if FORCE or not os.path.exists(pre + '_body_front.png'):
    sh([BLENDER, '-b', '-P', A('show_slices3d.py'), '--', sorted(glob.glob(os.path.join(MODEL_DIR, '*.glb')))[0], pre] + fits)
show(pre + '_body_front.png', pre + '_body_side.png', pre + '_body_back.png', width=300)

## 4. Build the real forearm (atlas bones + muscles warped in, closed skin) and move it with Newton flesh (bend the elbow, twist). Cut: pink skin, yellow sticky layer, red muscle, ivory bone

In [ ]:
FORCE = False; BEND, TWIST = 45, 60          # degrees
out = os.path.join(MODEL_DIR, 'anatomy'); limb = os.path.join(out, 'forearm_r_limb.npz'); sim = os.path.join(out, 'forearm_r_sim.npz')
if FORCE or not os.path.exists(limb):
    sh([PY, '-W', 'ignore', A('limb_build.py'), ATLAS + r'\atlas.npz', os.path.join(SLICES, 'forearm_r.npz'), os.path.join(out, 'forearm_r_fit.npz'), limb])
if FORCE or not os.path.exists(sim):          # ~7 min on this laptop
    sh([PYS, '-W', 'ignore', A('limb_sim.py'), limb, os.path.join(MODEL_DIR, 'body', 'skeleton.npz'), sim, str(BEND), str(TWIST)])
    sh([BLENDER, '-b', '-P', A('show_limb_sim.py'), '--', sim, os.path.join(out, 'forearm_sim')])
show(os.path.join(out, 'forearm_sim_*_a.png'), width=300); show(os.path.join(out, 'forearm_sim_*_b.png'), width=300)
show(os.path.join(out, 'forearm_sim_cut_*.png'), width=400)